# Maven Market — Data Modeling

## tl;dr

The cleaned Maven Market tables were converted into a star schema with two facts and four dimensions. The build preserved 269,720 sales lines and 7,087 return lines, produced no orphan keys, and reconciled to **$1,764,546.44 revenue** and **$1,052,818.78 gross product profit**.

The model also flags the 13 stores that can be used for a like-for-like comparison between 1997 and 1998.

## Context & Methods

The dashboard needs simple one-to-many relationships and shared dimensions across sales and returns. I used surrogate keys, flattened Region into Store, and kept sales and returns as separate facts because they do not share the same grain.

### Key assumptions

- A transaction row is a sales line, not an order.
- Product price and cost are copied into the sales fact when the model is built.
- Return rows cannot be assigned to customers.
- Statistical scaling is not required for BI measures stored in their original units.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.build_star_schema import (
    load_processed_tables,
    build_model_tables,
    validate_model,
    export_model,
)

pd.set_option("display.max_columns", 30)
PROJECT_ROOT

PosixPath('/mnt/data/maven-market-retail-intelligence-phase-05-data-modeling')

## Data

Load the processed tables from the cleaning stage and build the six model tables.

In [2]:
source_tables = load_processed_tables()
model_tables = build_model_tables(source_tables)
validation = validate_model(source_tables, model_tables)
export_model(model_tables, validation)

validation["status"]

'passed'

## Results

### Table profile

In [3]:
table_profile = pd.read_csv(PROJECT_ROOT / "reports" / "model_table_profile.csv")
table_profile

,table,grain,primary_key,rows,columns
0,dim_date,One row per calendar date,date_key,737,15
1,dim_customer,One row per customer,customer_key,10281,24
2,dim_product,One row per product,product_key,1560,13
3,dim_store,"One row per store, including region attributes",store_key,24,17
4,fact_sales,One cleaned source transaction line,sales_line_key,269720,16
5,fact_returns,One cleaned source return line,return_line_key,7087,8


### Relationship plan

All filters use a single direction from Dimension to Fact. The stock-date relationship is kept inactive.

In [4]:
relationships = pd.read_csv(PROJECT_ROOT / "reports" / "model_relationships.csv")
relationships

,from_table,from_column,to_table,to_column,cardinality,active_in_power_bi,filter_direction
0,dim_date,date_key,fact_sales,transaction_date_key,1:*,True,Single
1,dim_date,date_key,fact_sales,stock_date_key,1:*,False,Single
2,dim_date,date_key,fact_returns,return_date_key,1:*,True,Single
3,dim_product,product_key,fact_sales,product_key,1:*,True,Single
4,dim_product,product_key,fact_returns,product_key,1:*,True,Single
5,dim_customer,customer_key,fact_sales,customer_key,1:*,True,Single
6,dim_store,store_key,fact_sales,store_key,1:*,True,Single
7,dim_store,store_key,fact_returns,store_key,1:*,True,Single


### Reconciliation

In [5]:
pd.Series(validation["reconciliation"], name="value").to_frame()

,value
quantity_sold,833489.00
return_quantity,8289.00
revenue,1764546.44
product_cost_amount,711727.66
gross_product_profit,1052818.78
gross_profit_margin_pct,59.67
same_store_comparable_stores,13.00


### Sample sales fact rows

In [6]:
model_tables["fact_sales"].head()

,sales_line_key,transaction_line_id,transaction_date_key,stock_date_key,product_key,customer_key,store_key,quantity_sold,unit_retail_price,unit_product_cost,revenue,product_cost_amount,gross_product_profit,source_year,source_row_number,duplicate_candidate
0,1,TX-1997-000001,19970101,19961231,869,3449,6,5,2.12,0.91,10.60,4.55,6.05,1997,1,False
1,2,TX-1997-000002,19970101,19961231,1472,3449,6,3,2.20,0.90,6.60,2.70,3.90,1997,2,False
2,3,TX-1997-000003,19970101,19961228,76,3449,6,4,1.69,0.69,6.76,2.76,4.00,1997,3,False
3,4,TX-1997-000004,19970101,19961226,320,3449,6,3,3.26,1.08,9.78,3.24,6.54,1997,4,False
4,5,TX-1997-000005,19970101,19961225,4,3449,6,4,3.64,1.64,14.56,6.56,8.00,1997,5,False


### Validation checks

In [7]:
check_table = (
    pd.Series(validation["checks"], name="passed")
    .rename_axis("check")
    .reset_index()
)
check_table

,check,passed
0,dim_date_key_unique,True
1,dim_customer_key_unique,True
2,dim_product_key_unique,True
3,dim_store_key_unique,True
4,fact_sales_key_unique,True
5,fact_returns_key_unique,True
6,sales_row_count_preserved,True
7,return_row_count_preserved,True
8,sales_foreign_keys_complete,True
9,return_foreign_keys_complete,True


## Takeaways

- The model is ready to import into Power BI from `data/model/`.
- Sales can be filtered by Date, Customer, Product, and Store.
- Returns can be filtered by Date, Product, and Store, but not Customer.
- `dim_store[is_same_store_comparable]` should be used for like-for-like year comparisons.
- No Order or Channel dimension is included because the source does not contain those entities.
- The model validation passed without row loss or broken keys.